In [1]:
import rclpy
from rclpy.node import Node

from geometry_msgs.msg import PointStamped
from tf2_ros import Buffer, TransformListener
from tf2_geometry_msgs import do_transform_point

from depthai_ros_msgs.msg import TrackDetection2DArray

import math
from geometry_msgs.msg import Vector3

import numpy as np
from rclpy.time import Time


def rclpy_duration_to_float(duration: rclpy.time.Duration) -> float:
    """
    Convert a ros2 duration (seconds/nanoseconds) to a float.

    Args:
        time: time to convert

    Return:
        time (seconds) as a float
    """
    return float(duration.nanoseconds) / 1e9

In [2]:
class PersonEKF:

    def __init__(self):
        self.x = np.zeros((4,1))  # [x, y, vx, vy]
        self.P = np.eye(4) * 1.0

        # self.Q = np.eye(4) * 0.05   # process noise
        self.Q = np.eye(4) * 0.5   # process noise
        self.R = np.eye(2) * 0.1    # measurement noise

    def predict(self, dt):

        F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1,  0],
            [0, 0, 0,  1]
        ])

        self.x = F @ self.x
        self.P = F @ self.P @ F.T + self.Q

    def update(self, z):

        H = np.array([
            [1,0,0,0],
            [0,1,0,0]
        ])

        y = z - H @ self.x
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)

        self.x = self.x + K @ y
        I = np.eye(4)
        self.P = (I - K @ H) @ self.P

Using:

Topic: /human_relative

Message type: geometry_msgs/msg/Vector3

Vector3.x → distance
Vector3.y → steering angle (rad)
Vector3.z → unused (0.0)

In [3]:
class TrackToMapPoint(Node):

    def __init__(self):
        super().__init__('track_to_map_point')
        # TF buffer & listener
        self.tf_buffer = Buffer()
        self.tf_listener = TransformListener(self.tf_buffer, self)

        self.subscription = self.create_subscription(
            TrackDetection2DArray,
            '/track_detections',
            self.track_callback,
            10
        )

        self.publisher = self.create_publisher(
            PointStamped,
            '/human_position_map',
            10
        )
        
        self.human_pub = self.create_publisher(
            Vector3,
            '/human_relative',
            1
        )

        self.ekf = PersonEKF()

        # Call on_timer function every second
        self.timer = self.create_timer(0.5, self.on_timer)
        self.camera_to_map_transform = None
        self.map_to_robot_transform = None

    def on_timer(self):
        # ---- Lookup transform ----
        try:
            self.camera_to_map_transform = self.tf_buffer.lookup_transform(
                'map',
                'oakd_rgb_camera_optical_frame',
                rclpy.time.Time(),
                timeout=rclpy.duration.Duration(seconds=5.)
            )
        except Exception as ex:
            print("self.camera_to_map_transform", self.camera_to_map_transform)
            self.get_logger().warn(
                f'TF transform failed: {str(ex)}'
            )

        try:
            self.map_to_robot_transform = self.tf_buffer.lookup_transform(
                'base_link',        # target (robot)
                'map',              # source
                rclpy.time.Time(),
                timeout=rclpy.duration.Duration(seconds=0.2)
            )
        except Exception as ex:
            print("self.map_to_robot_transform", self.map_to_robot_transform)
            self.get_logger().warn(
                f'TF transform failed: {str(ex)}'
            )

    def apply_ekf_point(self, current_point):
        z = np.array([
            [current_point.point.x],
            [current_point.point.y]
        ])

        dt = 0.1
        self.ekf.predict(dt)
        self.ekf.update(z)

        # print("EKF result: ", self.ekf.x)
        test = current_point
        test.point.x = self.ekf.x[0][0]
        test.point.y = self.ekf.x[1][0]

        return test

    def track_callback(self, msg: TrackDetection2DArray):
        if self.camera_to_map_transform is None:
            return

        time1 = Time.from_msg(msg.header.stamp)
        time2 = Time.from_msg(self.map_to_robot_transform.header.stamp)

        diff_time = rclpy_duration_to_float(time1 - time2)
        if diff_time > 0.4:
            return
            
        print("\n\n ################## ")
        print("Track header:", time1)
        print("Transform header:", time2)
        print("Time difference: ", time1 > time2, rclpy_duration_to_float(time1 - time2))
        
        point_map = None
        for detection in msg.detections:
            tracking_id = detection.tracking_id
            if tracking_id != "0":
                continue
            
            for track in detection.results:              
                # ---- Create PointStamped in robot frame ----
                point_robot = PointStamped()
                point_robot.header.stamp = msg.header.stamp
                point_robot.header.frame_id = "oakd_rgb_camera_optical_frame" #  self.target_frame
    
                # ADAPT FIELD NAMES TO YOUR MESSAGE
                point_robot.point.x = track.pose.pose.position.x
                point_robot.point.y = track.pose.pose.position.y
                point_robot.point.z = track.pose.pose.position.z

                try:
                    # ---- Transform point ----
                    point_map = do_transform_point(point_robot, self.camera_to_map_transform)

                    ekf_point_map = self.apply_ekf_point(point_map)
                    ekf_point_map.header.frame_id = "map"
    
                    self.publisher.publish(ekf_point_map)

                except Exception as ex:
                    print("self.camera_to_map_transform", self.camera_to_map_transform)
                    self.get_logger().warn(
                        f'TF transform failed: {str(ex)}'
                    )

                # Transform human point into robot frame
                human_robot = do_transform_point(ekf_point_map, self.map_to_robot_transform)
        
                x = human_robot.point.x
                y = human_robot.point.y
        
                distance = math.sqrt(x*x + y*y)
                angle = math.atan2(y, x)   # LEFT positive
        
                out = Vector3()
                out.x = distance
                out.y = angle
                out.z = 0.0
        
                self.human_pub.publish(out)
        
                # # Transform human point into robot frame       
                # x = track.pose.pose.position.z
                # y = track.pose.pose.position.x
        
                # distance = math.sqrt(x*x + y*y)
                # # angle = math.atan2(y, x)   # LEFT positive

                # angle = np.arctan(x / (y + 1e-5))
        
                # out = Vector3()
                # out.x = distance
                # out.y = angle
                # out.z = 0.0
        
                # self.human_pub.publish(out)

        if point_map is None:
            return


In [ ]:
rclpy.init()

node = TrackToMapPoint()
rclpy.spin(node)

node.destroy_node() 
rclpy.shutdown()

self.camera_to_map_transform None


[WARN] [1771920388.759103818] [track_to_map_point]: TF transform failed: "map" passed to lookupTransform argument target_frame does not exist. 


self.map_to_robot_transform None


[WARN] [1771920388.970224359] [track_to_map_point]: TF transform failed: "base_link" passed to lookupTransform argument target_frame does not exist. 


self.camera_to_map_transform None


...

[WARN] [1771920488.409850106] [track_to_map_point]: TF transform failed: "base_link" passed to lookupTransform argument target_frame does not exist. 


self.camera_to_map_transform None


[WARN] [1771920493.420831141] [track_to_map_point]: TF transform failed: "map" passed to lookupTransform argument target_frame does not exist. 


self.map_to_robot_transform None


[WARN] [1771920493.635204660] [track_to_map_point]: TF transform failed: "base_link" passed to lookupTransform argument target_frame does not exist. 
